# 01 · Full-archive extraction on a cloud VM / Colab
The development laptop reads the public WB2 bucket at ≈1.9 MB/s; the ENS needs ≈240 MB of chunks per init (T6), i.e. ≈876 GB for 2018–2022. Run this on a machine in or near GCP (Colab or a small VM). It is resumable (one Parquet/npz per init) — copy `data/interim/fcst` and `data/interim/state` back afterwards.

In [ ]:
!git clone <your-remote>/purva-netra.git && cd purva-netra && pip -q install -e .
# IMD truth + masks are small; build them first so weights exist
!cd purva-netra && make truth

In [ ]:
# Split by year across parallel processes (each is independent and resumable)
import subprocess
procs = [subprocess.Popen(['scripts/extract_loop.sh', f'{y}-01-01', f'{y}-12-31'], cwd='purva-netra') for y in range(2018, 2023)]
procs += [subprocess.Popen(['.venv/bin/python','-m','purva_netra.extract_state', f'{y}-01-01', f'{y}-12-31'], cwd='purva-netra') for y in range(2018, 2023)]
[p.wait() for p in procs]